# Part III: Local review prototype

Student Review Studio serves the frozen P4 pipeline through a local Streamlit app. It estimates recorded Dropout at normal course-end from 18 enrolment fields. Production selection minimizes the existing nested training-only OOF log loss among B0/P1–P4, with exact ties by ID; P1B remains a sensitivity analysis. P4 (0.539483) and P2 (0.539911) are close; the difference does not demonstrate stable superiority. Neither Q4 results nor reserved outcomes selected the pipeline.

The bundled pipeline includes preprocessing and 100 shallow boosting trees (depth 2, leaf size 10, learning rate 0.1). Loading checks its SHA-256, input schema and class order. No fitting occurs during import, installation, prediction or startup.

The interface provides an individual form and CSV cohort review. Inputs must match the source coding dictionary and be available at enrolment. Missing, impossible and undocumented values are rejected; extrapolation and documented-but-untrained categories generate warnings. Partial cohorts require acknowledgment before capacity is calculated on valid rows. Uploaded names, outcomes and extra fields are ignored and not exported.

The synthetic examples below contain no real students or observed outcomes. Review uses floor(q*N) at 5%, 10%, 20%, with SHA256(1481:row_id) ties. IDs refer to 1-based input positions. Reordering tied records can change the selected list. This reuses the Q4 selector; it does not estimate intervention benefit or current-upload performance.

Run `python -m streamlit run app.py --server.address 127.0.0.1 --browser.gatherUsageStats false` from the project root for the interface. The cells below exercise the same prediction and export functions.

In [1]:
import sys
import io
import json
from pathlib import Path
import numpy as np
import pandas as pd
sys.path.insert(0, str(Path('src').resolve()))
from student_dropout import production

students = pd.read_csv('examples/synthetic_students.csv')
probabilities = production.predict(students)
expected = json.loads(Path('assets/parity_expected.json').read_text())
np.testing.assert_allclose(probabilities, expected['dropout_probability'], atol=1e-12, rtol=0)
print(f'Synthetic individual dropout probability: {probabilities[0]:.1%}')
print('All 20 synthetic predictions match the frozen reference.')

Synthetic individual dropout probability: 12.7%
All 20 synthetic predictions match the frozen reference.


In [2]:
validated = production.validate(students)
for share in (0.05, 0.10, 0.20):
    reviewed = production.review(validated, share=share)
    print(f'{share:.0%} capacity: {int(reviewed.selected_for_review.sum())} selected')
exported = production.export_csv(reviewed)
restored = pd.read_csv(io.BytesIO(exported))
assert len(restored) == 20 and restored.selected_for_review.sum() == 4
display(restored.head())

5% capacity: 1 selected
10% capacity: 2 selected
20% capacity: 4 selected


,row_id,status,errors,warnings,dropout_probability,selected_for_review,rank,model_version
0,10,valid,NaN,NaN,0.517009,True,1,P4-1.0.0-b5119c011186
1,12,valid,NaN,NaN,0.508861,True,2,P4-1.0.0-b5119c011186
2,11,valid,NaN,NaN,0.483321,True,3,P4-1.0.0-b5119c011186
3,8,valid,NaN,NaN,0.478339,True,4,P4-1.0.0-b5119c011186
4,18,valid,NaN,NaN,0.464453,False,5,P4-1.0.0-b5119c011186


## Validation and limits

The executable product tests cover prediction parity at absolute tolerance 1e-12, available-feature enforcement, deterministic ties and capacities, malformed input, unseen categories, partial-cohort acknowledgment, export reproducibility and Streamlit interactions. The browser smoke test additionally checks actual uploads and downloads. See README.md for commands; these are synthetic checks, not evidence of held-out predictive performance.

Class 0 combines Graduate and Enrolled at the recorded horizon; Enrolled does not mean eventual graduation. Course horizons differ. Earlier full-sample EDA indirectly explored the later reserved test set. Transport, prospective calibration and subgroup performance remain unverified. No individual uncertainty interval has been estimated. Q2 associations are not causal, and Q4 identifies recorded cases rather than prevented dropouts. The original tutoring/advising effect remains unidentified; an unidentified effect is not zero, and the numerical causal-effect/standard-error assignment requirement remains unmet.

Data and category definitions: Realinho et al. (2021), [UCI dataset 697](https://doi.org/10.24432/C5MC89), CC BY 4.0. No software license has been selected.